In [0]:
import pandas as pd
from datetime import date
from pyspark.sql.functions import col, lit, when

#Global

In [0]:
tb_moni = 'ia.tb_diamond_mod_monitoramento'

tb_entrada = 'diamond_hepatologia.hepatologia.tb_diamond_mod_hepatologia_entrada'
tb_saida = 'diamond_hepatologia.hepatologia.vw_diamond_mod_hepatologia'

tb_retorno = 'diamond_hepatologia.hepatologia.tb_diamond_mod_hepatologia_retorno'

tb_navegacao = 'diamond_hepatologia.hepatologia.tb_diamond_mod_hepatologia_navegacao'
tb_analise_retorno = 'diamond_hepatologia.hepatologia.vw_diamond_mod_hepatologia_analise_retorno'


#Criacao da tabela de monitoramento

In [0]:
df_entrada = spark.sql(f"""
    select 
        dt_execucao,
        emp_regional_unidade as nm_regional,
        COUNT(id_exame) AS qtd_entrada,
        emp_nome_unidade as nm_unidade,
        id_unidade
    from {tb_entrada}
    where emp_regional_unidade IN ('RJ','SP','BA')
    GROUP BY dt_execucao,emp_regional_unidade,emp_nome_unidade,id_unidade
    ORDER BY dt_execucao
""")

df_saida = spark.sql(f"""
    SELECT
        dt_execucao,
        emp_regional_unidade as nm_regional,
        COUNT(id_exame) AS qtd_saida,
        id_unidade
    FROM {tb_saida}
    WHERE emp_regional_unidade IN ('RJ','SP','BA')
    GROUP BY dt_execucao,emp_regional_unidade,id_unidade
    ORDER BY dt_execucao
""")

df_envio = spark.sql(f"""
    SELECT
        dt_execucao,
        emp_regional_unidade as nm_regional,
        COUNT(id_exame) AS qtd_envio,
        id_unidade
    FROM {tb_saida}
    WHERE fl_relevante = 1
      AND emp_regional_unidade IN ('RJ','SP','BA')
      AND nome_convenio not rlike r'(?i)(?<!\w)AMIL(?:\s*[-/().A-Za-z0-9]+)?'
      AND num_idade_paciente <= 80
    GROUP BY dt_execucao,emp_regional_unidade,id_unidade
    ORDER BY dt_execucao
""")

In [0]:
df_monitoramento = (
    df_entrada.alias("e")
        .join(df_saida.alias("s"), on=["dt_execucao", "nm_regional", "id_unidade"], how="left")
        .join(df_envio.alias("v"), on=["dt_execucao", "nm_regional", "id_unidade"], how="left")
        .select(
            "dt_execucao",
            "e.qtd_entrada",
            "s.qtd_saida",
            "v.qtd_envio",
            "e.nm_regional",
            "e.nm_unidade",
        )
)

In [0]:
df_monitoramento = (
    df_monitoramento
        .withColumn("qtd_entrada", col("qtd_entrada").cast("int"))
        .withColumn("qtd_saida", col("qtd_saida").cast("int"))
        .withColumn("qtd_envio", col("qtd_envio").cast("int"))
        .withColumn("nm_regional", col("nm_regional").cast("string"))
        .withColumn("nm_unidade", col("nm_unidade").cast("string"))
        .withColumn("nm_projeto", lit("Figado").cast("string"))
)

In [0]:
df_monitoramento.createOrReplaceTempView("vw_monitoramento_wrk_01")

In [0]:
amostra = spark.sql(f"""
select 
dt_execucao,
-- qtd_entrada,
-- qtd_saida,
sum(qtd_envio) as qtd_envio,
-- qtd_preenchido,
-- qtd_nao_preenchido,
-- qtd_navegacao,
-- nm_unidade,
nm_regional
from vw_monitoramento_wrk_01
where nm_projeto = "Figado" 
--and nm_unidade is not null
and nm_regional = "SP" 
and dt_execucao = "2026-02-09"
group by dt_execucao,nm_regional
"""
)
amostra.display()

In [0]:
%sql
MERGE INTO ia.tb_diamond_mod_monitoramento t
USING vw_monitoramento_wrk_01 s
ON  t.dt_execucao = s.dt_execucao
AND t.nm_unidade = s.nm_unidade
AND t.nm_regional = s.nm_regional
AND t.nm_projeto  = s.nm_projeto

WHEN NOT MATCHED THEN
  INSERT (
    dt_execucao,
    qtd_entrada,
    qtd_saida,
    qtd_envio,
    nm_regional,
    nm_unidade,
    nm_projeto
  )
  VALUES (
    s.dt_execucao,
    s.qtd_entrada,
    s.qtd_saida,
    s.qtd_envio,
    s.nm_regional,
    s.nm_unidade,
    s.nm_projeto
  )


In [0]:
# Use esse bloco apenas se a tabela de monitoramento estiver vazia(Drop Table)
# df_monitoramento.write.mode("append").saveAsTable(tb_moni)

#Criacao DF_complemento

In [0]:
df_preenchimento = spark.sql(f"""
    select 
        ret.dt_execucao,
        count(ret.id_exame) as qtd_preenchido,
        ent.emp_nome_unidade as nm_unidade,
        ent.emp_regional_unidade as nm_regional
    from {tb_retorno} as ret
    inner join  {tb_entrada} as ent
    on ret.id_exame = ent.id_exame
    group by ret.dt_execucao,ent.emp_nome_unidade,ent.emp_regional_unidade
    order by ret.dt_execucao
""")

df_navegacao = spark.sql(f"""
    select 
        ret.dt_execucao,
        count(ret.id_exame) as qtd_navegacao,
        ent.emp_nome_unidade as nm_unidade,
        ent.emp_regional_unidade as nm_regional
    from {tb_retorno} as ret
    inner join  {tb_entrada} as ent
    on ret.id_exame = ent.id_exame
    where cod_destino not like '%6%'
    group by ret.dt_execucao,ent.emp_nome_unidade,ent.emp_regional_unidade
    order by ret.dt_execucao
""")

In [0]:
df_retorno = (
    df_preenchimento.alias("p")
    .join(df_navegacao, on=["dt_execucao", "nm_regional","nm_unidade"], how="left")
    .withColumn("qtd_preenchido",when(col("qtd_preenchido").isNull(), lit(0)).otherwise(col("qtd_preenchido")))
    .withColumn("qtd_navegacao",when(col("qtd_navegacao").isNull(),lit(0)).otherwise(col("qtd_navegacao")))
    .select(
        "dt_execucao",
        "qtd_preenchido",
        "p.nm_regional",
        "qtd_navegacao",
        "nm_unidade"
        )
)

In [0]:
df_retorno = (
    df_retorno
        .withColumn("nm_regional", col("nm_regional").cast("string"))
        .withColumn("qtd_preenchido", col("qtd_preenchido").cast("int"))
        .withColumn("qtd_navegacao", col("qtd_navegacao").cast("int"))
        .withColumn("nm_unidade", col("nm_unidade").cast("string"))
)

In [0]:
df_retorno.createOrReplaceTempView("vw_monitoramento_retorno")

#Update e/ou insert

In [0]:
%sql
MERGE INTO ia.tb_diamond_mod_monitoramento t
USING vw_monitoramento_retorno s
ON t.dt_execucao = s.dt_execucao 
and t.nm_regional = s.nm_regional
and t.nm_unidade = s.nm_unidade
AND t.nm_projeto = "Figado"

WHEN MATCHED AND (
       coalesce(t.qtd_preenchido, 0) <> coalesce(s.qtd_preenchido, 0)
    OR coalesce(t.qtd_navegacao, 0) <> coalesce(s.qtd_navegacao, 0)
)
THEN UPDATE SET
    t.qtd_preenchido = coalesce(s.qtd_preenchido, 0),
    t.qtd_navegacao  = coalesce(s.qtd_navegacao, 0),
    t.qtd_nao_preenchido = coalesce(t.qtd_envio, 0) - coalesce(s.qtd_preenchido, 0)

WHEN NOT MATCHED
AND s.nm_regional IS NOT NULL
THEN INSERT (
    dt_execucao,
    qtd_preenchido,
    qtd_navegacao,
    qtd_nao_preenchido
)
VALUES (
    s.dt_execucao,
    coalesce(s.qtd_preenchido, 0),
    coalesce(s.qtd_navegacao, 0),
    0
)


#Criacao df_navegacao


In [0]:
df_relevante = spark.sql(f"""
select
        count(*) as qtd_relevante,
        nav.dt_enviado_navegacao as dt_execucao,
       
        ret.emp_nome_unidade as nm_unidade,
        ret.emp_regional_unidade as nm_regional
    
        from {tb_analise_retorno} as ret
        inner join {tb_navegacao} as nav
          on ret.id_predicao = nav.id_predicao
        where ret.emp_regional_unidade in ("SP","RJ","BA")

        group by 
          nav.dt_enviado_navegacao,
          ret.emp_nome_unidade,
          ret.emp_regional_unidade
        order by nav.dt_enviado_navegacao
      
""")

In [0]:
df_relevante = (
    df_relevante
        .withColumn("nm_unidade", col("nm_unidade").cast("string"))
        .withColumn("nm_regional", col("nm_regional").cast("string"))
        .withColumn("qtd_relevante", col("qtd_relevante").cast("int"))
)

In [0]:
df_relevante.createOrReplaceTempView("vw_monitoramento_relevante")

In [0]:
%sql
MERGE INTO ia.tb_diamond_mod_monitoramento t
USING vw_monitoramento_relevante s
ON t.dt_execucao = s.dt_execucao
AND t.nm_regional = s.nm_regional
AND t.nm_unidade = s.nm_unidade
AND t.nm_projeto = "Figado"

WHEN MATCHED
AND t.qtd_relevante IS NULL
AND s.qtd_relevante IS NOT NULL
THEN UPDATE SET
    t.qtd_relevante = s.qtd_relevante

In [0]:
dbutils.notebook().exit("Fim da execução!")

#Amostra

In [0]:
dbutils.widgets.text("dt_execucao", str(date.today()), "DATA")
dt_execucao = dbutils.widgets.get("dt_execucao")
print("data:", dt_execucao)

In [0]:
amostra = spark.sql(f"""
select 
dt_execucao,
qtd_entrada,
qtd_saida,
qtd_envio,
qtd_preenchido,
qtd_nao_preenchido,
qtd_navegacao,
nm_unidade,
nm_regional
from ia.tb_diamond_mod_monitoramento 
where nm_projeto = "Figado" 
-- and nm_unidade is not null
and nm_regional = "SP" 
and dt_execucao > '{dt_execucao}'
"""
)
amostra.display()

In [0]:
amostra = spark.sql(f"""
select 
-- *
dt_execucao,
-- -- -- qtd_entrada,
-- -- -- qtd_saida,
sum(qtd_envio) as qtd_envio,
-- -- -- qtd_preenchido,
-- -- -- qtd_nao_preenchido,
-- -- -- qtd_navegacao,
-- -- -- nm_unidade,
nm_regional
from ia.tb_diamond_mod_monitoramento 
where nm_projeto = "Figado" 
--and nm_unidade is not null
and nm_regional = "SP" 
and dt_execucao >= '{dt_execucao}'
-- and nm_unidade is null
group by dt_execucao,nm_regional
"""
)
amostra.display()

In [0]:
amostra = spark.sql(f"""
select 
dt_execucao,
sum(qtd_relevante) as qtd_relevante,
nm_regional
from ia.tb_diamond_mod_monitoramento 
where nm_projeto = "Figado" 
-- and nm_unidade is not null
and nm_regional = "RJ" 
and dt_execucao > '{dt_execucao}'
group by dt_execucao,nm_regional
"""
)
amostra.display()

In [0]:
df_relevante = spark.sql(f"""
select
        count(*) as total,
        -- ret.dt_execucao as dataExecucaoModelo,
        nav.dt_enviado_navegacao as dataNavegacao,

        -- ret.id_predicao as idPredicao,
        -- ret.id_exame as idExame,
       
        ret.emp_nome_unidade as nomeHospital,
        ret.emp_regional_unidade as regionalHospital
        
        -- date_format(date(ret.dt_exame), 'dd/MM/yyyy') as dataExame
    
        from {tb_analise_retorno} as ret
        inner join {tb_navegacao} as nav
          on ret.id_predicao = nav.id_predicao
        where ret.emp_regional_unidade in ("SP","RJ","BA")

        group by 
          nav.dt_enviado_navegacao,
          ret.emp_nome_unidade,
          ret.emp_regional_unidade
        order by nav.dt_enviado_navegacao
      
""")
    
df_relevante.display()

In [0]:
%sql
-- delete from ia.tb_diamond_mod_monitoramento 
-- where dt_execucao <= "2026-02-09"
-- and  nm_projeto = "Figado" 
-- and nm_unidade is null

In [0]:
# spark.sql(
#     f"""
#     CREATE TABLE IF NOT EXISTS {tb_moni_dev} (
#         dt_execucao DATE,
#         qtd_entrada INT,
#         qtd_saida INT,
#         qtd_envio INT,
#         qtd_navegacao INT,
#         qtd_preenchido INT,
#         qtd_nao_preenchido INT,
#         nm_regional STRING,
#         nm_projeto STRING,
#         qtd_relevante int,
#         nm_unidade string
#     )
#     USING DELTA
#     """
# )